# Mashariki Weather Example
This brief notebook demonstrates the use of MasharikiWeather for wind prediction. We obtain ERA5 wind speed data for Kenya and perform time series forecasting. This example is motivated by a similar example using [PeakWeather](https://github.com/MeteoSwiss/PeakWeather).

This example makes use of the [Torch Spatiotemporal (tsl)](https://torch-spatiotemporal.readthedocs.io/en/latest/) library for spatiotemporal data processing.

In [ ]:
import tsl
import torch
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

from masharikiweather.dataset import MasharikiWeatherDataset
from tsl.data import SpatioTemporalDataset

from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler
from tsl.nn.models.temporal import RNNModel

from tsl.metrics.torch import MaskedMAE, MaskedMAPE
from tsl.engines import Predictor


## Initialize the pipeline

Access tokens are needed to obtain the data.

In [ ]:
with open('mashariki_tokens.json', 'r') as f:
    key_data = json.load(f)

In [ ]:
ds = MasharikiWeatherDataset(
    repo_id="DeKUT-DSAIL/weather-data",
    token=key_data['huggingface dataset token'], 
    source_obs=["tahmo", "ghcnd"], # Fusing hourly and daily networks
    freq="h", 
    years=[2020, 2021, 2022, 2023, 2024]
)

## Get ERA5 data
We obtain the $u$ and $v$ components of wind velocity at 10m and extract a region in central Kenya.

In [ ]:
era5_data = ds.load_gridded_datasets(groups=["era5"],
                                     years=[2020])

In [ ]:
era5 = era5_data['era5']
wind_u = era5[['10m_u_component_of_wind']]
wind_v = era5[['10m_v_component_of_wind']]

In [ ]:
# limit to central Kenya
wind_u = wind_u.sel(lat=slice(0.4, -1.2), lon=slice(36, 38))
wind_v = wind_v.sel(lat=slice(0.4, -1.2), lon=slice(36, 38))

In [ ]:
%%time
wind_u = wind_u.compute()
wind_v = wind_v.compute()

### Visualisations

In [ ]:
wind1d_u = wind_u['10m_u_component_of_wind'].isel(lat=2, lon=0)
wind1d_v = wind_v['10m_v_component_of_wind'].isel(lat=2, lon=0)
wind1d_u.plot();

## Create a spatiotemporal dataset
We convert each grid point into a location and form a dataframe. We then leverage [(tsl)](https://torch-spatiotemporal.readthedocs.io/en/latest/) to split the wind velocity timeseries into sliding windows with respective forecast horizons. See [this explanation](https://torch-spatiotemporal.readthedocs.io/en/latest/usage/spatiotemporal_dataset.html).

In [ ]:
ds_stacked = wind_u.stack(location=("lat", "lon"))

df = (
    ds_stacked
    .drop_vars(["lat", "lon", "location"])   # 🔑 remove duplicates
    .to_dataframe()
    .reset_index()
)
df_pivot = df.pivot(index="time", columns="location", values="10m_u_component_of_wind")

In [ ]:
window_size = 12 
forecast_horizon = 12

torch_dataset = SpatioTemporalDataset(target=df_pivot,
                                      horizon=forecast_horizon,
                                      window=window_size,
                                      stride=1)
print(torch_dataset)

### Normalization and train-val-test split

In [ ]:


# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.2)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=64,
)
dm.setup()

In [ ]:
print(dm)

### RNN for wind speed forecasting

In [ ]:
model = RNNModel(input_size=1, output_size=1, horizon=torch_dataset.horizon)

In [ ]:
model

In [ ]:

loss_fn = MaskedMAE()

metrics = {'mae': MaskedMAE(),
           'mape': MaskedMAPE(),
           'mae_at_15': MaskedMAE(at=2),  # '2' indicates the third time step,
                                          # which correspond to 15 minutes ahead
           'mae_at_30': MaskedMAE(at=5),
           'mae_at_60': MaskedMAE(at=11)}

# setup predictor
predictor = Predictor(
    model=model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 0.001},    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=metrics                # metrics to be logged during train/val/test
)

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath='logs',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
)

trainer = pl.Trainer(max_epochs=100,
                     logger=None,
                     accelerator='cpu',
                     limit_train_batches=100,  # end an epoch after 100 updates
                     callbacks=[checkpoint_callback])

trainer.fit(predictor, datamodule=dm)

In [ ]:
predictor.load_model(checkpoint_callback.best_model_path)
predictor.freeze()

trainer.test(predictor, datamodule=dm);
